# Minnesota County Housing Map
**Data Source:** U.S. Census Bureau — ACS 5-Year Estimates (2022)  
**State:** Minnesota  
**Author:** Scott A. May

This notebook pulls county-level housing data from the Census API, joins it to Census TIGER shapefiles, and renders an interactive choropleth map of Minnesota counties color-coded by median home value.



In [46]:
import requests
import pandas as pd
import geopandas as gpd
import folium
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")
STATE = "27"  # Minnesota
YEAR = 2022
VARIABLES = "NAME,B25077_001E,B25002_003E,B25003_002E,B25003_003E"

print("Configuration loaded ✓")


Configuration loaded ✓


## Step 1: Pull Data from Census API
Pulling county-level housing data for Minnesota.

In [47]:
url = (
    f"https://api.census.gov/data/{YEAR}/acs/acs5"
    f"?get={VARIABLES}"
    f"&for=county:*"
    f"&in=state:{STATE}"
    f"&key={API_KEY}"
)

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data[1:], columns=data[0])

print(f"Pulled {len(df)} counties ✓")
df.head()

Pulled 87 counties ✓


,NAME,B25077_001E,B25002_003E,B25003_002E,B25003_003E,state,county
0,"Aitkin County, Minnesota",222100,7400,5751,1046,27,001
1,"Anoka County, Minnesota",302300,4143,107811,26579,27,003
2,"Becker County, Minnesota",249600,5539,11049,3085,27,005
3,"Beltrami County, Minnesota",203900,3533,12114,5756,27,007
4,"Benton County, Minnesota",229300,1006,10842,5470,27,009


## Step 2: Clean and Shape the Data
Renaming columns, converting to numeric, and creating a FIPS code for joining to geographic data.

In [48]:
df = df.rename(columns={
    "B25077_001E": "median_home_value",
    "B25002_003E": "vacant_units",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
})

for col in ["median_home_value", "vacant_units", "owner_occupied", "renter_occupied"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["FIPS"] = df["state"] + df["county"]

print(f"Data cleaned ✓ — {len(df)} counties")
df.head()

Data cleaned ✓ — 87 counties


,NAME,median_home_value,vacant_units,owner_occupied,renter_occupied,state,county,FIPS
0,"Aitkin County, Minnesota",222100,7400,5751,1046,27,001,27001
1,"Anoka County, Minnesota",302300,4143,107811,26579,27,003,27003
2,"Becker County, Minnesota",249600,5539,11049,3085,27,005,27005
3,"Beltrami County, Minnesota",203900,3533,12114,5756,27,007,27007
4,"Benton County, Minnesota",229300,1006,10842,5470,27,009,27009


## Step 3: Load County Boundaries
Pulling Census TIGER shapefiles for Minnesota county boundaries and joining to housing data.

In [49]:
shapefile_url = (
    "https://www2.census.gov/geo/tiger/GENZ2022/shp/"
    "cb_2022_us_county_500k.zip"
)

print("Loading shapefile...")
gdf = gpd.read_file(shapefile_url)
gdf = gdf[gdf["STATEFP"] == STATE].copy()
gdf["FIPS"] = gdf["STATEFP"] + gdf["GEOID"].str[-3:]

merged = gdf.merge(df, on="FIPS", how="left")
merged = merged.dropna(subset=["median_home_value"])

print(f"Shapefile loaded and joined ✓ — {len(merged)} counties")

Loading shapefile...
Shapefile loaded and joined ✓ — 87 counties


## Step 4: Build the Interactive Map
Rendering a choropleth map color-coded by median home value with county-level tooltips.

In [50]:
center = [46.5, -94.0]  # Hardcoded center of Minnesota

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=merged.__geo_interface__,
    data=merged,
    columns=["FIPS", "median_home_value"],
    key_on="feature.properties.FIPS",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Median Home Value ($)",
    bins=6,
).add_to(m)

folium.GeoJson(
    merged.__geo_interface__,
    style_function=lambda x: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME_x", "median_home_value", "vacant_units",
                "owner_occupied", "renter_occupied"],
        aliases=["County", "Median Home Value ($)",
                 "Vacant Units", "Owner Occupied", "Renter Occupied"],
        localize=True,
    )
).add_to(m)

m.save("housing_opportunity_map.html")
print("Map saved ✓ — open housing_opportunity_map.html in your browser")

Map saved ✓ — open housing_opportunity_map.html in your browser
